# BIT-S4-DD8

This notebook is a thin entry point over the reusable `satchange` package. A fresh **Run All** performs only configuration, construction, parameter-count, and forward-smoke checks. Dataset preparation and tiny-overfit training are opt-in flags below.

In [ ]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset

from satchange import BCEDiceLoss, fit_binary_change_model
from satchange.data import create_dataset_dataloader, prepare_levir_cd
from satchange.models import count_model_parameters, create_change_model
from satchange.utils import load_config

In [ ]:
current_directory = Path.cwd().resolve()
project_root = (
    current_directory.parent
    if current_directory.name == "notebooks"
    else current_directory
)
if not (project_root / "pyproject.toml").is_file():
    raise FileNotFoundError("Run this notebook from the project root or notebooks directory.")

config_path = project_root / "configs" / "bit.yaml"
config = load_config(config_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(config["project"]["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config["project"]["seed"])

model = create_change_model(
    model_name=config["model"]["name"],
    parameters=config["model"]["parameters"],
).to(device)
print("Config:", config_path)
print("Device:", device)
print("Parameters:", count_model_parameters(model))

In [ ]:
image_size = config["data"]["image_size"]
before = torch.rand(1, 3, image_size, image_size, device=device)
after = torch.rand(1, 3, image_size, image_size, device=device)
model.eval()
with torch.no_grad():
    logits = model(before, after)
print("Forward output:", logits.shape)

## Optional dataset setup

Set `RUN_DATA_SETUP = True` only when LEVIR-CD preparation and loader construction are intended.

In [ ]:
RUN_DATA_SETUP = False

if RUN_DATA_SETUP:
    dataset_root = prepare_levir_cd(config=config, project_root=project_root)
    train_dataset, train_loader = create_dataset_dataloader(
        dataset_root=dataset_root,
        split="train",
        image_size=config["data"]["image_size"],
        batch_size=config["data"]["batch_size"],
        num_workers=config["data"]["num_workers"],
        shuffle=True,
        pin_memory=torch.cuda.is_available(),
        seed=config["project"]["seed"],
    )
    _, validation_loader = create_dataset_dataloader(
        dataset_root=dataset_root,
        split="val",
        image_size=config["data"]["image_size"],
        batch_size=config["data"]["batch_size"],
        num_workers=config["data"]["num_workers"],
        pin_memory=torch.cuda.is_available(),
    )
    print("Dataset root:", dataset_root)
    print("Train/validation batches:", len(train_loader), len(validation_loader))
else:
    print("Dataset setup skipped; set RUN_DATA_SETUP=True to enable it.")

In [ ]:
loss_config = config["training"]["loss"]
loss_function = BCEDiceLoss(
    bce_weight=loss_config["bce_weight"],
    dice_weight=loss_config["dice_weight"],
    pos_weight=loss_config["pos_weight"],
).to(device)
print(loss_function)

## Optional deterministic tiny overfit

This diagnostic deliberately fits and validates on the same four training samples. It creates fresh model, loss, optimizer, and loaders inside the guard and writes to a separate smoke-test directory.

In [ ]:
RUN_TINY_OVERFIT = False

if RUN_TINY_OVERFIT:
    dataset_root = prepare_levir_cd(config=config, project_root=project_root)
    overfit_source, _ = create_dataset_dataloader(
        dataset_root=dataset_root,
        split="train",
        image_size=config["data"]["image_size"],
        batch_size=2,
        num_workers=0,
    )
    tiny_indices = range(min(4, len(overfit_source)))
    tiny_dataset = Subset(overfit_source, tiny_indices)
    generator = torch.Generator().manual_seed(config["project"]["seed"])
    tiny_train_loader = DataLoader(
        tiny_dataset, batch_size=2, shuffle=True, generator=generator
    )
    tiny_validation_loader = DataLoader(tiny_dataset, batch_size=2, shuffle=False)

    overfit_model = create_change_model(
        model_name=config["model"]["name"],
        parameters=config["model"]["parameters"],
    ).to(device)
    overfit_loss = BCEDiceLoss(
        bce_weight=loss_config["bce_weight"],
        dice_weight=loss_config["dice_weight"],
        pos_weight=loss_config["pos_weight"],
    ).to(device)
    overfit_optimizer = torch.optim.AdamW(
        overfit_model.parameters(),
        lr=config["training"]["learning_rate"],
        weight_decay=config["training"]["weight_decay"],
    )
    fit_binary_change_model(
        model=overfit_model,
        train_loader=tiny_train_loader,
        validation_loader=tiny_validation_loader,
        loss_function=overfit_loss,
        optimizer=overfit_optimizer,
        device=device,
        number_of_epochs=20,
        thresholds=[round(value / 100, 2) for value in range(5, 96, 5)],
        output_root=project_root / "artifacts" / "smoke_tests" / "bit",
        model_name=config["model"]["name"],
        run_name=f"tiny_overfit_seed_{config['project']['seed']}",
        config=config,
    )
else:
    print("Tiny overfit skipped; set RUN_TINY_OVERFIT=True to enable it.")